In [1]:
import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
train_df = pd.read_csv('./data/assignment-2-train.csv')
train_df

,x,y
0,10,300
1,30,500
2,50,400
3,60,700
4,70,500
5,90,700
6,100,600


In [3]:
test_df = pd.read_csv('./data/assignment-2-test.csv')
test_df

,x,y
0,20,500
1,40,600
2,90,600


In [4]:
class LinearRegressionModel:

    def __init__(self, learning_rate: float = 1e-3, epoch: int = 100):

        self.weights: np.ndarray[float] = None
        self.bias = 0

        self.lr = learning_rate
        self.epoch = epoch

        self.rmse_history: list[float] = []


    def predict(self, x: float | np.ndarray[float]) -> float:
        return np.dot(x, self.weights) + self.bias
    
    
    def RMSE(self, y_true: np.ndarray[float], y_pred: np.ndarray[float]) -> float:
        return np.sqrt(np.mean((y_pred - y_true) ** 2))
    

    def fit(self, x: pd.DataFrame, y: pd.Series) -> None:
        
        n_samples = x.shape[0]
        n_features = x.shape[1]

        if n_samples == 0:
            raise ValueError('Training data is empty')
        
        x_ndarray = x.to_numpy()
        y_ndarray = y.to_numpy()

        self.weights = np.zeros(n_features) 
        self.bias = 0

        for _ in tqdm.tqdm(range(self.epoch)):
            
            predictions = self.predict(x_ndarray)
            
            current_rmse = self.RMSE(y_ndarray, predictions)
            self.rmse_history.append(current_rmse)

            errors = predictions - y_ndarray

            dw = (1 / n_samples) * np.dot(x_ndarray.T, errors)
            db = (1 / n_samples) * np.sum(errors)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

In [5]:
X_train = train_df[['x']]
y_train = train_df['y']

X_test = test_df[['x']]
y_test_actual = test_df['y']

In [6]:
model = LinearRegressionModel(learning_rate=1e-4, epoch=100000)
model.fit(X_train, y_train)

100%|██████████| 100000/100000 [00:01<00:00, 81814.30it/s]


In [7]:
print(f"Weights (Slopes): {model.weights}")
print(f"Bias (Intercept): {model.bias}")

Weights (Slopes): [4.08154318]
Bias (Intercept): 280.8448204383827


In [8]:
predictions = model.predict(X_test.to_numpy())

pd.DataFrame({
	'X': test_df['x'],
	'Actual_y': y_test_actual,
	'Predicted_y': predictions
})

,X,Actual_y,Predicted_y
0,20,500,362.475684
1,40,600,444.106548
2,90,600,648.183707


In [9]:
rmse = model.RMSE(y_test_actual, predictions)
print(f"\nFinal RMSE on Test Set: {rmse:.4f}")

print(f"Learned Weight (Slope): {model.weights}")
print(f"Learned Bias (Intercept): {model.bias:.4f}")


Final RMSE on Test Set: 123.2036
Learned Weight (Slope): [4.08154318]
Learned Bias (Intercept): 280.8448
